In [0]:
%run /Repos/dung_nguyen_hoang@mfcgd.com/Utilities/Functions

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
import pandas as pd
from pyspark.sql.window import Window

In [0]:
start_dt = '2025-01-01'
end_dt   = '2025-04-30'
mthend_sht = end_dt[:4] + end_dt[5:7]

# Retrieving latest exchange rate (for VND to USD conversion)
exrt_df = spark.sql(f"""
with xrt as (
select  cast(XCHNG_RATE as int) ex_rate, to_date(FR_EFF_DT) as EFF_DT,
        row_number() over (partition by XCHNG_RATE_TYP order by FR_EFF_DT DESC) rn
from    vn_published_cas_db.texchange_rates
where   XCHNG_RATE_TYP='U'
    and FR_CRCY_CODE='78'
    and to_date(FR_EFF_DT) <= '{end_dt}'
qualify rn=1
) select ex_rate from xrt
""")

ex_rate = exrt_df.collect()[0][0]
del exrt_df

### Load all policies successfully reached out for DD+45

In [0]:
# Load from BCS logs
loc_path = f"/mnt/lab/vn/project/scratch/adhoc/Premium_reminder_call_logs_2504.xlsx"
sheet_name = "Sheet1"
cell_pos = "A1"
successful_calls = spark.read.format(
    "com.crealytics.spark.excel"
).option(
    "dataAddress", f"{sheet_name}!{cell_pos}"
).option(
    "header", "true"
).option(
    "treatEmptyValuesAsNulls", "true"
).option(
    "inferSchema", "true"
).option(
    "addColorColumns", "False"
).load(loc_path)

successful_calls = successful_calls.withColumn(
    "FinalStatus",
    F.when(F.substring(F.col("CallStatus"),1,1).isin("2","3","4"), "A - Success")
    .when(F.substring(F.col("CallStatus"),1,1) == "1", "B - Fail")
    .otherwise("C - Not call")
)

In [0]:
# successful_calls.groupBy(
#     "FinalStatus",
#     "CallStatus"
# ).agg(
#     F.count("pol_num").alias("call_count"),
#     F.countDistinct("pol_num").alias("pol_count")
# ).orderBy(
#     "FinalStatus",
#     "CallStatus"
# ).display()

In [0]:
successful_calls_pmt = spark.sql("""
with full_trxn as (
select  POL_NUM, TRXN_AMT, TRXN_DT, REASN_CODE
from    vn_published_cas_db.ttrxn_histories_partial
where   REASN_CODE in ('301', '302','304', '306', '310', '501', '503', '505')
union
select  POL_NUM, TRXN_AMT, TRXN_DT, REASN_CODE
from    vn_published_cas_db.ttrxn_histories
where   REASN_CODE in ('301', '302','304', '306', '310', '501', '503', '505')
), po_list as (
select  cpl.CLI_NUM as PO_NUM, pol.POL_NUM, pol.PLAN_CODE_BASE, pol.POL_STAT_CD, pol.PMT_MODE,
        pol.PD_TO_DT, pol.MODE_PREM, pol.POL_ISS_DT, pol.AGT_CODE
from    vn_published_cas_db.tpolicys pol
inner join
        vn_published_cas_db.tclient_policy_links cpl on pol.POL_NUM = cpl.POL_NUM and cpl.LINK_TYP='O' and cpl.REC_STATUS='A'
)
select  distinct 
        pol.po_num,
        cso.pol_num, 
        pol.plan_code_base, fld2.FLD_VALU_DESC_ENG as policy_status, 
        case pol.pmt_mode
          when '12' then 'a. Yearly'
          when '06' then 'b. Half-yearly'
          when '03' then 'c. Quarterly'
          when '01' then 'd. Monthly' 
        end as pmt_mode,
        to_date(pol.pd_to_dt) as PTD,
        cso.br_name,
        to_date(cso.CloseDate) as closed_date, to_date(txn.TRXN_DT) as txn_dt,
        substr(to_date(cso.CloseDate),1,7) as call_month,
        datediff(txn.TRXN_DT, cso.CloseDate) as interval,
        concat_ws('-',txn.REASN_CODE, fld.FLD_VALU_DESC_ENG) txn_desc,
        cast(txn.TRXN_AMT as int) as txn_amt, 
        cast(mode_prem*cast(pmt_mode as int)/12 as int) as total_ape,
        case
            when floor(months_between('2025-04-30', pol.pol_iss_dt)/12) < 1 then 'a. <1yr'
            when floor(months_between('2025-04-30', pol.pol_iss_dt)/12) < 2 then 'b. 1-2yr'
            when floor(months_between('2025-04-30', pol.pol_iss_dt)/12) < 3 then 'c. 2-3yr'
            when floor(months_between('2025-04-30', pol.pol_iss_dt)/12) < 6 then 'd. 3-5yr'
            when floor(months_between('2025-04-30', pol.pol_iss_dt)/12) < 8 then 'e. 5-7yr'
            when floor(months_between('2025-04-30', pol.pol_iss_dt)/12) <11 then 'f. 7-10yr'
            else 'g. >10yr'
        end as pol_tenure,
        pol.agt_code,
        cso.FinalStatus
from    po_list pol
inner join
        {successful_calls} cso on (pol.POL_NUM = cso.pol_num)
left join
        full_trxn txn on (
          cso.pol_num = txn.POL_NUM and 
          txn.TRXN_DT >= cso.CloseDate
        )
left join vn_published_cas_db.tfield_values fld on txn.REASN_CODE = fld.FLD_VALU and fld.FLD_NM = 'REASN_CODE'
left join vn_published_cas_db.tfield_values fld2 on pol.POL_STAT_CD = fld2.FLD_VALU and fld2.FLD_NM = 'STAT_CODE'
""", 
successful_calls = successful_calls)

In [0]:
# successful_calls = spark.sql(f"""
# with full_trxn as (
# select  POL_NUM, TRXN_AMT, TRXN_DT, REASN_CODE
# from    vn_published_cas_db.ttrxn_histories_partial
# where   REASN_CODE in ('301', '302','304', '306', '310', '501', '503', '505')
# union
# select  POL_NUM, TRXN_AMT, TRXN_DT, REASN_CODE
# from    vn_published_cas_db.ttrxn_histories
# where   REASN_CODE in ('301', '302','304', '306', '310', '501', '503', '505')
# )  
# select  distinct 
#         --origin, Requester_Role__c as requester, Subject, 
#         --cso.CaseNumber, crm.Final_Results_Full_Desc,
#         Policy_Number__c as pol_num, 
#         pol.plan_code_base, pol.pol_stat_cd, 
#         case pol.pmt_mode
#           when '12' then 'Yearly'
#           when '06' then 'Half-yearly'
#           when '03' then 'Quarterly'
#           when '01' then 'Monthly' 
#         end as pmt_mode,
#         to_date(pol.pd_to_dt) as PTD,
#         Branch_Name__c as br_name,
#         to_date(substr(ClosedDate,1,10)) as closed_date, to_date(txn.TRXN_DT) as txn_dt,
#         concat_ws('-',txn.REASN_CODE, fld.FLD_VALU_DESC_ENG) txn_desc,
#         cast(txn.TRXN_AMT as int) as txn_amt, 
#         cast(mode_prem*cast(pmt_mode as int)/12 as int) as total_ape
# from    vn_published_cas_db.tpolicys pol
# inner join
#         vn_published_sfdc_easyclaims_db.`case` cso on (pol.POL_NUM = cso.Policy_Number__c)
# left join
#         full_trxn txn on (
#           cso.Policy_Number__c = txn.POL_NUM and 
#           to_date(txn.TRXN_DT) >= to_date(substr(ClosedDate,1,10))
#         )
# left join vn_published_cas_db.tfield_values fld on txn.REASN_CODE = fld.FLD_VALU and fld.FLD_NM = 'REASN_CODE'
# left join vn_aa_reports.case_results_mapping crm on cso.Final_Results__c = crm.Final_Results__c
# where   cso.Status = 'Closed'
#   and   cso.Subject = '09 - Call overdue 45 days policies'
#   and   cso.Origin = 'Outbound call'
#   -- Select only "Successful" cases
#   and   cso.Final_Results__c in ('11','12','13','15','21','23','24','25','27','28','30','33','35','45','57','60','61')
#   and   to_date(substr(ClosedDate,1,10)) between '{start_dt}' and '{end_dt}'
# """)

# record_count = successful_calls.count()
# pol_count = successful_calls.select("pol_num").distinct().count()
# print(record_count, pol_count)

In [0]:
successful_calls_pmt = successful_calls_pmt.withColumn(
    "paid+55",
    F.when(((F.col("interval") >= 0) & 
            (F.col("interval") <=10)), 1)
    .otherwise(0)
).withColumn(
    "paid+60",
    F.when(((F.col("interval") > 10) & 
            (F.col("interval") <=15)), 1)
    .otherwise(0)
).withColumn(
    "paid>60", 
    F.when(
        F.col("interval") > 10, 1)
    .otherwise(0))


successful_calls_pmt = successful_calls_pmt.withColumn(
    "final_paid_ind",
    F.when(F.col("paid+55") == 1, 'a. paid+55')
     .when(F.col("paid+60") == 1, "b. paid+60")
     .when(F.col("paid>60") == 1, "c. paid>60")
     .otherwise(F.lit("d. no payment"))
)

In [0]:
# Reorder the payments by call dates and its earliest txn
window_spec = Window.partitionBy(
    "call_month",
    "pol_num",
).orderBy(
    F.col("closed_date").asc(),  # Get the earliest date first
    F.col("interval").asc()      # Get the smallest interval first
)

successful_calls_pmt = successful_calls_pmt.withColumn("row_num", F.row_number().over(window_spec))
successful_calls_pmt_dedup = successful_calls_pmt.filter(F.col("row_num") == 1).drop("row_num")

In [0]:
# successful_calls_pmt_dedup.limit(5).display()

In [0]:
lapse_decile = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/lapse/pre_lapse_deployment/lapse_mthly/lapse_score.parquet"
).select(
    F.col("pol_num").cast("string"), 
    F.col("month_snapshot").cast("string"), 
    F.col("premium_due").cast("string"),
    F.col("lapse_score").cast("float"), 
    F.col("decile").cast("int")
).filter(
    F.substring(F.col("month_snapshot"),1,4).isin("2024","2025")
).drop(
    "premium_due", "lapse_score"
).withColumnRenamed(
    "decile", "lapse_decile"
)

# Retain the latest score from all the snapshots
window_spec = Window.partitionBy("pol_num").orderBy(F.col("month_snapshot").desc())
lapse_decile = lapse_decile.withColumn("row_num", F.row_number().over(window_spec))
lapse_decile = lapse_decile.filter(F.col("row_num") == 1).drop("row_num","month_snapshot")

In [0]:
# Derive product holding from inforce base (as of latest date)
inf_cus_base = spark.sql(f"""
with prod_list as (
select  PLAN_CODE, INS_TYP
        , case when fld.FLD_VALU_DESC='Investment' then
                case when pln.PLAN_CODE like 'RUV%' then 'Investment-ILP'
                    when pln.PLAN_CODE like 'UL%' then 'Investment-UL'
                    else 'Investment-Others' end
               when fld.FLD_VALU_DESC='Whole Life' then
                case when pln.PLAN_CODE not like '%ED99' then 'Endowment'
                    else 'Whole Life' end
               when pln.PLAN_CODE in ('FDB01','BIC01','BIC02','BIC03','BIC04') then 'Digital'
               when pln.PLAN_CODE in ('BHC9I','CA360','CI360','CN360','CX360','PN001') then 'Micro-Activator'
              else fld.FLD_VALU_DESC end
         as PROD_TYPE
from    vn_published_cas_db.tplans pln inner join
        vn_published_cas_db.tfield_values fld on pln.INS_TYP = fld.FLD_VALU and fld.FLD_NM='INS_TYP'
where   1=1
  and   pln.CVG_TYP = 'B'
), cus_tenure as (
select  cpl.CLI_NUM as PO_NUM, to_date(min(pol.POL_ISS_DT)) as FRST_ISS_DT
from    vn_published_cas_db.tpolicys pol 
inner join
        vn_published_cas_db.tclient_policy_links cpl on pol.POL_NUM = cpl.POL_NUM and cpl.LINK_TYP='O' and cpl.REC_STATUS='A' 
where   pol.POL_STAT_CD not in ('A','N','R','X')
group by cpl.CLI_NUM
), inf_cus as (
select  -- holding RUV
        distinct cpl.CLI_NUM as PO_NUM
        , collect_set(pln.PROD_TYPE) as LIST_OF_PRODUCT_TYPES
        , collect_set(rpl.PROD_TYP) as LIST_OF_RIDER_TYPES
        , cast(sum(pol.MODE_PREM*cast(PMT_MODE as int)/12)/{ex_rate} as decimal(11,2)) as TOTAL_APE_USD
from    vn_published_cas_db.tpolicys pol 
inner join
        vn_published_cas_db.tclient_policy_links cpl on pol.POL_NUM = cpl.POL_NUM and cpl.LINK_TYP='O' and cpl.REC_STATUS='A' 
inner join
        vn_published_ams_db.tams_agents agt on pol.AGT_CODE = agt.AGT_CODE 
left join
        vn_published_cas_db.tcoverages rid on pol.POL_NUM = rid.POL_NUM and rid.CVG_TYP='R' and rid.CVG_STAT_CD in ('1','2','3','5','7','9') 
left join 
        prod_list pln on pol.PLAN_CODE_BASE = pln.PLAN_CODE 
left join
        vn_published_cas_db.tplans rpl on rid.PLAN_CODE = rpl.PLAN_CODE    
where     1=1
    and     pol.POL_STAT_CD not in ('A','N','R','X')                            -- in-force or Maturing customer
    --and     agt.comp_prvd_num in ('01','02','04','08','96','97','98')   -- Agency's compro (inc. orphan)
group by  cpl.CLI_NUM
)
select  a.*,
        case 
            when floor(months_between('{end_dt}', b.FRST_ISS_DT)/12) < 1 then 'a. <1yr'
            when floor(months_between('{end_dt}', b.FRST_ISS_DT)/12) < 2 then 'b. 1-2yr'
            when floor(months_between('{end_dt}', b.FRST_ISS_DT)/12) < 3 then 'c. 2-3yr'
            when floor(months_between('{end_dt}', b.FRST_ISS_DT)/12) < 6 then 'd. 3-5yr'
            when floor(months_between('{end_dt}', b.FRST_ISS_DT)/12) < 8 then 'e. 5-7yr'
            when floor(months_between('{end_dt}', b.FRST_ISS_DT)/12) <11 then 'f. 7-10yr'
            else 'g. >10yr'
        end as cus_tenure
from    inf_cus a
inner join
        cus_tenure b on a.PO_NUM = b.PO_NUM
--where     a.TOTAL_APE_USD > 0
""")

# Step 1: Create indicator columns for product types
product_holding = inf_cus_base.withColumn(
    "ILP_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-ILP"), "Y")
    .otherwise("N")
).withColumn(
    "UL_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Investment-UL"), "Y")
    .otherwise("N")
).withColumn(
    "ENDOWMENT_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Endowment"), "Y")
    .otherwise("N")
).withColumn(
    "WHOLE_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Whole Life"), "Y")
    .otherwise("N")
).withColumn(
    "TERM_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Term Life"), "Y")
    .otherwise("N")
).withColumn(
    "DIGITAL_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Digital"), "Y")
    .otherwise("N")
).withColumn(
    "MICRO_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Micro-Activator"), "Y")
    .otherwise("N")
).withColumn(
    "OTH_IND", 
    F.when(F.array_contains(F.col("LIST_OF_PRODUCT_TYPES"), "Accident"), "Y")
    .otherwise("N")
)

# Step 2: Explode LIST_OF_RIDER_TYPES into dynamic columns
unique_rider_types = product_holding.select(F.explode(F.col("LIST_OF_RIDER_TYPES")).alias("rider")).distinct().collect()
rider_type_columns = [rider.rider for rider in unique_rider_types]

for rider_type in rider_type_columns:
    product_holding = product_holding.withColumn(
        f"{rider_type}_IND",
        F.when(F.array_contains(F.col("LIST_OF_RIDER_TYPES"), rider_type), "Y").otherwise("N")
    )

product_holding = product_holding.drop('LIST_OF_PRODUCT_TYPES','LIST_OF_RIDER_TYPES')

In [0]:
# Add `rider_ind` and `product_type` flag
new_cols = ({
    'rider_ind': F.greatest(F.col('cancer_rider_ind'),
                            F.col('add_rider_ind'),
                            F.col('tpd_rider_ind'),
                            F.col('tp_rider_ind'),
                            F.col('mc_rider_ind'),
                            F.col('ci_rider_ind'),
                            F.col('hc_rider_ind')),

    'product_type': F.when((F.col('ilp_ind') == 'Y') &
                           (F.greatest(F.col('ul_ind'),
                                     F.col('endowment_ind'),
                                     F.col('whole_ind'),
                                     F.col('term_ind'),
                                     F.col('digital_ind'),
                                     F.col('micro_ind'),
                                     F.col('oth_ind')) == 'N'),
                           'ILP only')
                    .when((F.col('ul_ind') == 'Y') &
                           (F.greatest(F.col('ilp_ind'),
                                     F.col('endowment_ind'),
                                       F.col('whole_ind'),
                                       F.col('term_ind'),
                                       F.col('digital_ind'),
                                       F.col('micro_ind'),
                                       F.col('oth_ind')) == 'N'),
                           'UL only')
                    .when((F.col('endowment_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Endowment only')
                    .when((F.col('whole_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Whole Life only')
                    .when((F.col('term_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Term Life only')
                    .when((F.col('digital_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('micro_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Digital only')
                    .when((F.col('micro_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                     F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('oth_ind')) == 'N'),
                          'Micro only')
                    .when((F.col('oth_ind') == 'Y') &
                          (F.greatest(F.col('ilp_ind'),
                                      F.col('ul_ind'),
                                      F.col('endowment_ind'),
                                      F.col('whole_ind'),
                                      F.col('term_ind'),
                                      F.col('digital_ind'),
                                      F.col('micro_ind')) == 'N'),
                          'Accident only')
                    .when(F.greatest(F.col('ilp_ind'),
                                    F.col('ul_ind'),
                                    F.col('endowment_ind'),
                                    F.col('whole_ind'),
                                    F.col('term_ind'),
                                    F.col('digital_ind'),
                                    F.col('micro_ind'),
                                    F.col('oth_ind')) == 'Y', 'Multiple products'),
})

rider_types_col = (
    F.when(F.col("rider_ind") == "N", "z. No active rider")
    .when(
        (F.col("rider_ind") == "Y") & (F.col("hc_rider_ind") == "Y") &
        (F.col("tpd_rider_ind") == "N") & (F.col("add_rider_ind") == "N") &
        (F.col("tp_rider_ind") == "N") & (F.col("cancer_rider_ind") == "N") &
        (F.col("ci_rider_ind") == "N") & (F.col("mc_rider_ind") == "N"),
        "a. HCR Only"
    )
    .when(
        (F.col("rider_ind") == "Y") & (F.col("mc_rider_ind") == "Y") &
        (F.col("tpd_rider_ind") == "N") & (F.col("add_rider_ind") == "N") &
        (F.col("tp_rider_ind") == "N") & (F.col("cancer_rider_ind") == "N") &
        (F.col("ci_rider_ind") == "N") & (F.col("hc_rider_ind") == "N"),
        "b. MC Only"
    )
    .when(
        (F.col("rider_ind") == "Y") & (F.col("ci_rider_ind") == "Y") &
        (F.col("tpd_rider_ind") == "N") & (F.col("add_rider_ind") == "N") &
        (F.col("tp_rider_ind") == "N") & (F.col("cancer_rider_ind") == "N") &
        (F.col("hc_rider_ind") == "N") & (F.col("mc_rider_ind") == "N"),
        "c. CI Only"
    )
    .when(
        (F.col("rider_ind") == "Y") & 
        ((F.col("hc_rider_ind") == "Y") |
         (F.col("mc_rider_ind") == "Y") |
         (F.col("ci_rider_ind") == "Y")) &
        (F.col("tpd_rider_ind") == "N") & (F.col("add_rider_ind") == "N") &
        (F.col("tp_rider_ind") == "N") & (F.col("cancer_rider_ind") == "N"),
        "d. Other single rider"
    )
    .when(
        (F.col("rider_ind") == "Y") & (F.col("hc_rider_ind") == "Y") &
        ((F.col("tpd_rider_ind") == "Y") | (F.col("add_rider_ind") == "Y") |
         (F.col("tp_rider_ind") == "Y") | (F.col("cancer_rider_ind") == "Y") |
         (F.col("ci_rider_ind") == "Y") | (F.col("mc_rider_ind") == "Y")),
        "e. Multiple w/ HCR"
    )
    .when(
        (F.col("rider_ind") == "Y") & (F.col("mc_rider_ind") == "Y") &
        ((F.col("tpd_rider_ind") == "Y") | (F.col("add_rider_ind") == "Y") |
         (F.col("tp_rider_ind") == "Y") | (F.col("cancer_rider_ind") == "Y") |
         (F.col("ci_rider_ind") == "Y") | (F.col("hc_rider_ind") == "Y")),
        "f. Multiple w/ MC"
    )
    .when(
        (F.col("rider_ind") == "Y") & (F.col("ci_rider_ind") == "Y") &
        ((F.col("tpd_rider_ind") == "Y") | (F.col("add_rider_ind") == "Y") |
         (F.col("tp_rider_ind") == "Y") | (F.col("cancer_rider_ind") == "Y") |
         (F.col("hc_rider_ind") == "Y") | (F.col("mc_rider_ind") == "Y")),
        "g. Multiple w/ CI"
    )
    .otherwise("h. Multiple other")
)

product_holding = product_holding.withColumns(new_cols)
product_holding = product_holding.withColumn(
      "rider_types",
      rider_types_col
).drop(
      "cancer_rider_ind", "tpd_rider_ind", "add_rider_ind", "tp_rider_ind",
      "hc_rider_ind", "ci_rider_ind", "mc_rider_ind"
)

product_holding = product_holding.toDF(*[col.lower() for col in product_holding.columns])

check_dup(product_holding, "po_num")

In [0]:
# Load latest Agency Structure from MIS
loc_path = f"/mnt/lab/vn/project/scratch/adhoc/loc_to_psm_mapping_{mthend_sht}.xlsx"
sheet_name = "Structure"
cell_pos = "A1"
loc_to_psm = spark.read.format(
    "com.crealytics.spark.excel"
).option(
    "dataAddress", f"{sheet_name}!{cell_pos}"
).option(
    "header", "true"
).option(
    "treatEmptyValuesAsNulls", "true"
).option(
    "inferSchema", "true"
).option(
    "addColorColumns", "False"
).load(
    loc_path
)

# Only use the part below when confirmed/approved by Business team
agent_list = spark.sql(f"""
select  AGT_CODE as agt_code,
        case when stat_cd = '01' then '1.Inforce'
             when stat_cd = '99' then '4.Terminated'
        end as agt_status,
        case when trmn_dt is not null then '4.Terminated'
             when trmn_dt is null then
             case
                when comp_prvd_num in ('01','02','08','96')     then '1.Inforce'
                when comp_prvd_num = '04'                       then '2.Collector'
                when comp_prvd_num in ('97','98')               then '3.SM'
                else '5.Not-Agency'
            end
        end as agt_rltnshp,
        CASE
            WHEN mpro_title IS NOT NULL THEN
                CASE
                    WHEN mdrt_desc IS NOT NULL THEN 'a. MDRT'
                    ELSE 
                    CASE
                        WHEN mpro_title = 'P' THEN 'b. Platinum'
                        WHEN mpro_title = 'G' THEN 'c. Gold'
                        WHEN mpro_title = 'S' THEN 'd. Silver'
                    END
                END
            ELSE 'e. Normal'
        END AS mpro_title,
        agt.LOC_CODE loc_cd
from    vn_published_ams_db.tams_agents agt
where   1 = 1
    --and MPRO_TITLE in ('P','G','S')
    --and to_date(MPRO_TITLE_EFF_DT) <= '{end_dt}'
    --and TRMN_DT is null
""")

par_df = spark.read.format("parquet").load(
    f"/mnt/lab/vn/project/cpm/datamarts/TPARDM_MTHEND/image_date={end_dt}"
).select(
    "agt_cd", "last_mth_pol", "last_3m_pol", "last_9m_pol"
).withColumnRenamed(
    "agt_cd", "agt_code"
).dropDuplicates()

agent_list = agent_list.join(
    par_df, 
    "agt_code", 
    "left"
).join(
    loc_to_psm,
    "loc_cd",
    "left"
)

agent_list = agent_list.withColumn(
    "agt_actvness",
    F.when(F.col("last_mth_pol") > 0, "1m active")
    .when(F.col("last_3m_pol") > 0, "3m active")
    .when(F.col("last_9m_pol") > 0, "9m active")
    .otherwise("SA")
).select(
    "agt_code", "loc_cd", "mpro_title", "psm_name_8", "psm_name_9", "agt_rltnshp", "agt_actvness"
).fillna(
    {"psm_name_8": "Open",
     "psm_name_9": "Open"
     }
)

check_dup(agent_list, "agt_code")

### Merge all metrics to successful call logs

In [0]:
final = successful_calls_pmt_dedup.join(
    lapse_decile,
    on="pol_num",
    how="left"
).join(
    product_holding,
    on="po_num",
    how="left"
).join(
    agent_list,
    on="agt_code",
    how="left"
)


### Write to CSV for download

In [0]:
final.toPandas().to_csv(
    "/dbfs/mnt/lab/vn/project/scratch/adhoc/premium_call_base_details.csv",
    index=False,
    header=True
)